# Marabou

## Marabou

Marabou is a complete neural network verifier from Stanford/Hebrew University. It supports:
- Piecewise-linear networks (ReLU, max-pooling)
- Properties specified as linear constraints on inputs and outputs
- Query language for encoding complex properties
- Native support for convolutional networks

Marabou uses an SMT-style approach with DPLL(T) enumeration of neuron activation patterns, with specialized splitting heuristics for neural networks.

## ACAS Xu
The ACAS Xu system provides airborne collision avoidance advisories. Stanford trained 45 neural networks to replace a lookup table. Marabou was used to verify 10 safety properties including: 'if the intruder is directly ahead and approaching, the network should not recommend 'clear of conflict''. Several properties were violated, leading to fixes.

In [ ]:
import numpy as np

# Simulate Marabou-style query for a safety property
# Property: if input in region A, output must be in region B

np.random.seed(42)

class MarabouQuery:
    def __init__(self):
        self.input_constraints = []  # (feature_idx, lo, hi)
        self.output_constraints = []  # (output_idx, op, bound)
        # op: 'ge' (>=), 'le' (<=), 'eq' (==)
    
    def add_input_bound(self, idx, lo, hi):
        self.input_constraints.append((idx, lo, hi))
    
    def add_output_property(self, idx, op, bound):
        self.output_constraints.append((idx, op, bound))
    
    def describe(self):
        print('Marabou Query:')
        print('  Input constraints:')
        for idx, lo, hi in self.input_constraints:
            print(f'    x[{idx}] in [{lo}, {hi}]')
        print('  Output properties:')
        for idx, op, b in self.output_constraints:
            print(f'    output[{idx}] {op} {b}')

# ACAS Xu Property 1 (simplified):
# If far away and heading away from intruder -> output advisory should be 'clear of conflict'
q = MarabouQuery()
q.add_input_bound(0, 0.6, 1.0)   # rho: far away (normalized)
q.add_input_bound(1, -0.5, 0.5)  # theta: roughly centered
q.add_input_bound(2, 0.4, 1.0)   # psi: heading away
q.add_input_bound(3, 0.0, 0.3)   # v_own: slow
q.add_input_bound(4, 0.0, 0.3)   # v_int: slow
q.add_output_property(0, '<=', 3.9911)  # clear of conflict

q.describe()

# Simplified verifier: random sampling (not complete, but illustrative)
def W1(): return np.random.randn(10, 5)
def W2(): return np.random.randn(5, 10)
def W3(): return np.random.randn(5, 5)
def b1(): return np.random.randn(10)
def b2(): return np.random.randn(5)
def b3(): return np.random.randn(5)

np.random.seed(0)
w1, w2, w3 = W1(), W2(), W3()
bb1, bb2, bb3 = b1(), b2(), b3()

def net_acas(x):
    h1 = np.maximum(0, w1 @ x + bb1)
    h2 = np.maximum(0, w2 @ h1 + bb2)
    return w3 @ h2 + bb3

violations = 0
n_tested = 10000
for _ in range(n_tested):
    x = np.array([
        np.random.uniform(0.6, 1.0),
        np.random.uniform(-0.5, 0.5),
        np.random.uniform(0.4, 1.0),
        np.random.uniform(0.0, 0.3),
        np.random.uniform(0.0, 0.3),
    ])
    out = net_acas(x)
    if out[0] > 3.9911:
        violations += 1

print(f'\nEmpirical check ({n_tested} samples): {violations} violations found')
print('(Marabou would either prove 0 violations or find the exact counterexample)')
